# cuOpt VRP Formulation Extensions

## Purpose

This notebook is the routing companion to
`02-Autoclave-Formulation-Extensions.ipynb`.

It is a **model-choice atlas**, not another long drill-path walkthrough. Use it to answer:

- Which VRP family matches the requirement?
- Which cuOpt routing feature represents it?
- What new data is required?
- Does the requirement change data, constraints, objective, or problem class?
- Can the requirement remain in routing, or does it require MIP or a hybrid model?

Companion notebooks:

- `03b-Drill-Path-Routing-Workshop.ipynb` teaches one complete routing story.
- `05-VRP-Runnable-Comparisons.ipynb` executes six representative variants.

## Setup and Version Check

The atlas targets the installed cuOpt 26.06 Python routing API. The code cells verify that
the referenced methods are present in this environment.

In [1]:
import pandas as pd
import cuopt
from cuopt import routing

print(f"cuOpt version: {cuopt.__version__}")

cuOpt version: 26.06.00


## First Principle: VRP Variants Are Compositions

There is no useful finite list in which every VRP variant is isolated from every other
variant. A practical formulation is assembled from:

1. **Route structure** — one vehicle, many vehicles, depots, open or closed routes.
2. **Order requirements** — mandatory, optional, paired, timed, or vehicle-specific.
3. **Resource limits** — capacity, time, cost, breaks, or eligibility.
4. **Objective choices** — distance, time, prize, fleet cost, or balance.

For example, a capacitated pickup-and-delivery problem with time windows combines capacity,
paired orders, multiple vehicles, service timing, and route cost.

## Practical VRP Family Menu

The table below distinguishes named problem families from modifiers that compose with them.

In [2]:
family_catalog = pd.DataFrame(
    [
        ("TSP", "one vehicle; all orders required", "fleet size 1", "03b baseline"),
        ("VRP", "multiple vehicles", "fleet size > 1", "assign and sequence work"),
        ("CVRP", "vehicle capacity limits", "add_capacity_dimension", "tool or payload limits"),
        ("VRPTW", "order time windows", "set_order_time_windows", "operation availability"),
        ("CVRPTW", "capacity plus time windows", "compose both features", "payload and timing"),
        ("PDP", "paired pickup and delivery", "set_pickup_delivery_pairs", "paired transport"),
        ("PDPTW", "paired work plus time windows", "compose PDP and timing", "timed transport"),
        ("Multi-depot VRP", "vehicles start or return at different locations", "set_vehicle_locations", "multiple robot homes"),
        ("Open VRP", "vehicle need not return to its start", "drop return-trip behavior", "finish at last task"),
        ("Heterogeneous VRP", "vehicles have different costs or capabilities", "vehicle types and matrices", "different robot profiles"),
        ("Prize-collecting VRP", "some orders may be skipped", "set_order_prizes", "optional work by value"),
    ],
    columns=["family", "defining feature", "cuOpt pattern", "factory interpretation"],
)
display(family_catalog)

,family,defining feature,cuOpt pattern,factory interpretation
0,TSP,one vehicle; all orders required,fleet size 1,03b baseline
1,VRP,multiple vehicles,fleet size > 1,assign and sequence work
2,CVRP,vehicle capacity limits,add_capacity_dimension,tool or payload limits
3,VRPTW,order time windows,set_order_time_windows,operation availability
4,CVRPTW,capacity plus time windows,compose both features,payload and timing
5,PDP,paired pickup and delivery,set_pickup_delivery_pairs,paired transport
6,PDPTW,paired work plus time windows,compose PDP and timing,timed transport
7,Multi-depot VRP,vehicles start or return at different locations,set_vehicle_locations,multiple robot homes
8,Open VRP,vehicle need not return to its start,drop return-trip behavior,finish at last task
9,Heterogeneous VRP,vehicles have different costs or capabilities,vehicle types and matrices,different robot profiles


## Extension 1: One Robot to Multiple Robots

**Business question:** Is there one route, or must the solver assign operations across a
fleet and sequence each route?

**Model change:** `n_fleet` increases. The fleet size is normally an upper bound;
cuOpt prioritizes vehicle count before route cost unless a minimum is requested.

**Relevant API:** `DataModel(..., n_fleet, ...)` and
`set_min_vehicles()`.

## Extension 2: Capacity — CVRP and CVRPTW

**Business question:** Does each robot or vehicle have a limited quantity it can carry,
process, or support?

**Data required:** integer demand per order and integer capacity per vehicle.

**Relevant API:** `add_capacity_dimension(name, demand, capacity)`.

Multiple capacity dimensions can represent independent limits such as tool slots, payload,
or material units. Capacity plus time windows creates CVRPTW.

## Extension 3: Time Windows and Service Times

**Business question:** When may an operation begin, and how long does service take?

**Data required:** earliest and latest arrival, travel-time matrix, and service duration in
consistent units.

**Relevant API:**

- `add_transit_time_matrix()`
- `set_order_time_windows()`
- `set_order_service_times()`
- `set_vehicle_time_windows()`

Time windows constrain feasibility. The primary cost matrix can still optimize distance
while the transit-time matrix enforces timing.

## Extension 4: Pickup and Delivery

**Business question:** Must one operation occur before a paired operation on the same route?

**Data required:** pickup order indices, matching delivery indices, and often positive
pickup demand with negative delivery demand.

**Relevant API:** `set_pickup_delivery_pairs()`.

In the installed 26.06 API, all non-depot orders in a PDP model must participate in pairs;
mixed paired and ordinary orders are not supported in one PDP call.

## Extension 5: Heterogeneous Fleet and Eligibility

**Business question:** Do robots move at different speeds, incur different costs, or have
different qualifications?

**Data required:** vehicle type, type-specific cost or transit matrices, and any allowed
vehicle lists for restricted orders.

**Relevant API:**

- `set_vehicle_types()`
- multiple `add_cost_matrix(..., vehicle_type)` calls
- `add_order_vehicle_match()`
- `add_vehicle_order_match()`

## Extension 6: Optional Work and Prizes

**Business question:** May some orders be omitted when their value does not justify the
route cost?

**Data required:** prize or value per order.

**Relevant API:**

- `set_order_prizes()`
- `set_objective_function()` with `PRIZE` and `COST`

Prize collection changes all-required routing into a select-and-route problem. Objective
weights must use compatible scales so business value and travel cost are compared
deliberately.

## Extension 7: Depots, Open Routes, and Route Limits

| Requirement | cuOpt pattern |
| --- | --- |
| different robot homes | `set_vehicle_locations()` |
| no required return trip | vehicle drop-return-trip behavior |
| maximum route cost | `set_vehicle_max_costs()` |
| maximum route duration | `set_vehicle_max_times()` |
| vehicle availability | `set_vehicle_time_windows()` |
| fixed activation cost | `set_vehicle_fixed_costs()` |
| required fleet floor | `set_min_vehicles()` |

These features change route structure or fleet use without changing the problem into MIP.

## Extension 8: Breaks, Charging, and Maintenance

**Business question:** Must a robot or operator stop during an allowed interval, possibly at
a restricted set of locations?

**Relevant API:** homogeneous break dimensions, per-vehicle breaks, and break locations.

A charging or maintenance event fits a routing break only when it behaves like a stop on a
route. Decisions that couple charging resources, production schedules, or facility-wide
capacity may require a separate scheduling model or hybrid approach.

## Extension 9: Objective Choices

The installed routing objectives include:

- `COST`
- `TRAVEL_TIME`
- `VARIANCE_ROUTE_SIZE`
- `VARIANCE_ROUTE_SERVICE_TIME`
- `PRIZE`
- `VEHICLE_FIXED_COST`

`set_objective_function()` builds a weighted combination. Weights must be
interpreted carefully because each objective can use different units and scales.

In [3]:
objective_catalog = pd.DataFrame(
    [
        (objective.name, int(objective.value))
        for objective in routing.Objective
    ],
    columns=["objective", "numeric_value"],
)
display(objective_catalog)

,objective,numeric_value
0,COST,0
1,TRAVEL_TIME,1
2,VARIANCE_ROUTE_SIZE,2
3,VARIANCE_ROUTE_SERVICE_TIME,3
4,PRIZE,4
5,VEHICLE_FIXED_COST,5


## Extension 10: Solver Workflow Features

These capabilities change how routing data or solutions are produced rather than defining
a named VRP family:

- **Initial solutions** seed the search with known routes.
- **Rerouting** replans after completed or changed work.
- **Batch solving** solves multiple independent models.
- **Distance engine / waypoint matrices** derive route costs from a motion or network graph.

They belong in the operational workflow layer of the solution architecture.

## Installed API Availability

This check prevents the atlas from describing methods absent from the installed environment.

In [4]:
referenced_methods = [
    "add_break_dimension",
    "add_capacity_dimension",
    "add_cost_matrix",
    "add_order_vehicle_match",
    "add_transit_time_matrix",
    "add_vehicle_break",
    "add_vehicle_order_match",
    "set_min_vehicles",
    "set_objective_function",
    "set_order_locations",
    "set_order_prizes",
    "set_order_service_times",
    "set_order_time_windows",
    "set_pickup_delivery_pairs",
    "set_vehicle_fixed_costs",
    "set_vehicle_locations",
    "set_vehicle_max_costs",
    "set_vehicle_max_times",
    "set_vehicle_time_windows",
    "set_vehicle_types",
]

api_availability = pd.DataFrame(
    {
        "method": referenced_methods,
        "available_in_installed_DataModel": [
            hasattr(routing.DataModel, method)
            for method in referenced_methods
        ],
    }
)
display(api_availability)

if not api_availability["available_in_installed_DataModel"].all():
    raise RuntimeError("A referenced cuOpt method is unavailable.")

,method,available_in_installed_DataModel
0,add_break_dimension,True
1,add_capacity_dimension,True
2,add_cost_matrix,True
3,add_order_vehicle_match,True
4,add_transit_time_matrix,True
5,add_vehicle_break,True
6,add_vehicle_order_match,True
7,set_min_vehicles,True
8,set_objective_function,True
9,set_order_locations,True


## Composition Recipes

| Requirement bundle | Resulting routing formulation |
| --- | --- |
| one robot + all required | TSP-style routing |
| multiple robots + all required | basic VRP |
| multiple robots + capacity | CVRP |
| multiple robots + time windows | VRPTW |
| capacity + time windows | CVRPTW |
| paired work + time windows + capacity | PDPTW |
| different robot types + eligibility | heterogeneous VRP |
| optional work + prize and cost objectives | prize-collecting VRP |
| multiple homes + open routes | multi-depot open VRP |

Features can be combined only when their data contracts and business meanings are
compatible.

## The Routing-to-MIP Boundary

Keep the problem in cuOpt routing when decisions remain local to routes:

- which vehicle serves an order;
- order sequence;
- route timing;
- vehicle capacity;
- paired pickup and delivery;
- route-level eligibility, breaks, and limits.

Consider MIP or a hybrid architecture when decisions couple routes to shared planning
choices:

- production batches or machine schedules;
- shared tools, crews, docks, chargers, or staging capacity across routes;
- logical either/or decisions spanning multiple routes or days;
- facility-wide resource calendars;
- arbitrary cross-route precedence;
- integrated routing and reactive-rework scheduling.

The installed 26.06 routing solve does not support arbitrary order precedence. Use time
windows or pickup-delivery pairing only when those features truthfully represent the
requirement.

## Final Decision Checklist

1. How many vehicles are available, and must all be used?
2. Are all orders mandatory?
3. Does any vehicle carry a limited quantity?
4. Are there order or vehicle time windows?
5. Are any orders paired?
6. Are vehicle costs, speeds, or capabilities different?
7. Do vehicles start or finish at different locations?
8. Are route duration, distance, breaks, or eligibility constrained?
9. Which objective reflects the real operational goal?
10. Does any decision couple multiple routes to a shared planning resource?

Use `05-VRP-Runnable-Comparisons.ipynb` to see six representative changes
executed on controlled teaching data.

Further reading:
[NVIDIA cuOpt routing Python API](https://docs.nvidia.com/cuopt/user-guide/26.06/cuopt-python/routing/routing-api.html)